# Training 3 different RNNs

In this notebook I will train 3 different RNNs, one simple RNN, one LSTM and one GRU.

In [ ]:
import numpy as np
import pandas as pd
import os
import random
from RNN_models import *
import torch
from torch import optim


device = torch.device("cpu")

data_folder = "../../MainProject/data/mediapipe_ugly_recordings"

In [ ]:
def load_array(data_path, n_frames=30, n_features=66) -> tuple[torch.tensor, torch.tensor]:
    """
    Loads a csv-file and reshapes it into (-1, n_frames, n_features)

    Args:
        data_path: Path to the data to load
        n_frames: How many rows of data to use in shaping
        n_features: How many columns of data to use in shaping
    
    Returns:
        X, Y: The data in shape (-1, n_frames, n_features), The target as a 1darray
    """
    data = pd.read_csv(data_path)
    y = data["target"].values
    x = data.drop(columns=["target"]).values
    return torch.tensor(x.reshape(-1, n_frames, n_features), dtype=torch.float32), torch.tensor(y)


def split_csvfiles(datafolder, random_seed, training_prop, validation_prop):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)

    random.seed(random_seed)
    random.shuffle(csv_files)

    train_n = int(len(csv_files) * training_prop)
    val_n = int(len(csv_files) * validation_prop)

    # Split
    if validation_prop == 0:
        train_files = csv_files[:train_n]
        test_files = csv_files[train_n:]

        return train_files, test_files

    else:
        train_files = csv_files[:train_n]
        val_files = csv_files[train_n: train_n + val_n]
        test_files = csv_files[train_n + val_n:]

        return train_files, val_files, test_files

In [ ]:
train_files, test_files = split_csvfiles(data_folder, random_seed=42, training_prop=0.9, validation_prop=0)

pairs = []
for file in train_files:
    path = os.path.join(data_folder, file)
    x, y = load_array(path)
    x = x.to(device)
    y = y.to(device)
    pairs.append((x, y))

print(f"Total numer of files to train on: {len(train_files)}")
print(f"Total number of files to test on: {len(test_files)}")

### Initialize the models

In [ ]:
def train_model(params: dict) -> tuple[nn.Module, dict]:
    model = params["model_type"](params["input_size"], params["hidden_size"], params["depth"], params["num_of_classification_labes"]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=params["lr"])
    loss_func = nn.CrossEntropyLoss().to(device)
    prev_loss = np.inf
    streak = 0
    loss_increase = False
    loss_history = []
    early_stopping = False

    for epoch in range(params["epochs"]):
        total_loss = 0

        for X, Y in pairs:

            output = model(X)

            loss = loss_func(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        
        loss_history.append(total_loss)

        # Check if loss increased this epoch
        if total_loss > prev_loss:
            loss_increase = True
        # Count number of times loss increase in a row
        if loss_increase:
            streak += 1
        else:
            streak = 0
        # Early stopping
        if streak == params["patience"]:
            early_stopping = True
            break
        
        print(total_loss, )

    stats = {
        "loss_history": loss_history,
        "early_stopping": early_stopping
        }
    return model, stats

In [ ]:
params_RNN = {
    "model_type": SimpleRNNModel,
    "input_size": 66,
    "hidden_size": 64,
    "depth": 3,
    "num_of_classification_labels": 4,
    "lr": 0.0001,
    "optimizer": "Adam",
    "loss": "CrossEntropyLoss",
    "patience": 3,
    "epochs": 100
}

params_RNN = {
    "model_type": LSTMModel,
    "input_size": 66,
    "hidden_size": 64,
    "depth": 3,
    "num_of_classification_labels": 4,
    "lr": 0.0001,
    "optimizer": "Adam",
    "loss": "CrossEntropyLoss",
    "patience": 3,
    "epochs": 100
}

params_RNN = {
    "model_type": GRUModel,
    "input_size": 66,
    "hidden_size": 64,
    "depth": 3,
    "num_of_classification_labels": 4,
    "lr": 0.0001,
    "optimizer": "Adam",
    "loss": "CrossEntropyLoss",
    "patience": 3,
    "epochs": 100
}

RNN, RNN_stats = train_model(params=params_RNN)
LSTM, LSTM_stats = train_model(params=params_LSTM)
GRU, GRU_stats = train_model(params=params_GRU)